<a href="https://colab.research.google.com/github/porekhov/drug_design_2024/blob/main/qsar_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q rdkit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

In [ ]:
def add_rdkit_descriptors(
    df: pd.DataFrame,
    smiles_col: str = "smiles",
    keep_invalid_as_nan: bool = True,
) -> pd.DataFrame:
    """
    Add RDKit descriptors to a pandas DataFrame:
      - logP: Crippen MolLogP
      - MW: molecular weight
      - TPSA: topological polar surface area

    Parameters
    ----------
    df : pd.DataFrame
        Input table containing a SMILES column.
    smiles_col : str
        Name of the column with SMILES strings.
    keep_invalid_as_nan : bool
        If True, invalid SMILES get NaN values.
        If False, invalid SMILES raise a ValueError.

    Returns
    -------
    pd.DataFrame
        Copy of the input table with added columns: logP, MW, TPSA.
    """

    if smiles_col not in df.columns:
        raise KeyError(f"Column '{smiles_col}' not found in DataFrame.")

    result = df.copy()

    logp_values = []
    mw_values = []
    tpsa_values = []

    for i, smi in result[smiles_col].items():
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None

        if mol is None:
            if keep_invalid_as_nan:
                logp_values.append(float("nan"))
                mw_values.append(float("nan"))
                tpsa_values.append(float("nan"))
                continue
            else:
                raise ValueError(f"Invalid SMILES at index {i}: {smi}")

        logp_values.append(Crippen.MolLogP(mol))
        mw_values.append(Descriptors.MolWt(mol))
        tpsa_values.append(rdMolDescriptors.CalcTPSA(mol))

    result["logP"] = logp_values
    result["MW"] = mw_values
    result["TPSA"] = tpsa_values

    return result

In [ ]:
# reading a dataset with skin permeability coefficients
# data taken from https://www.nature.com/articles/s41598-021-89587-5#MOESM1
df = pd.read_csv('logKp_dataset.csv', sep = '\t')

df = add_rdkit_descriptors(df)

logp = df['logP']
logkp = df['logKp']

In [ ]:
# 1D case: correlation between skin permeability and logP
slope, intercept = np.polyfit(logp, logkp, 1)
corr, pval = pearsonr(logp, logkp)

fig1 = plt.figure()
ax = fig1.add_subplot(111)

plt.scatter(logp, logkp, marker = 'o', c = 'r')

plt.plot(np.array([-4, 4]), slope*np.array([-4, 4]) + intercept, color='k',
         ls =':', label = 'ρ = '+str(round(corr,2)))

plt.xlabel('logP')
plt.ylabel('logKp')
plt.legend()
plt.show()

In [ ]:
# 3D case: adding 2 more descriptors to the linear model: MW and TPSA

mw, tpsa = df['MW'], df['TPSA']

params = np.vstack((logp, mw, tpsa, np.ones(20))).T
print(params)
# finding the least-square solution
koefs = np.linalg.lstsq(params, logkp, rcond=None)[0]
print(koefs)
# Calculating prediction based on the linear regression model
logkp_pred = np.dot(params, koefs)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(logkp, logkp_pred, marker = '^', color = 'g', lw = 0, ms = 10)
ax.plot([-5, 0], [-5, 0], lw = 3, color = 'gray')

ax.set_xlabel('Model data')
ax.set_ylabel('Experimental data')

plt.show()

In [ ]:
# mean square error
print(np.sqrt(np.mean((logkp_pred - logkp)**2)))
# Pearson’s correlation coefficient can be calculated directly
print(np.sum((logkp - np.mean(logkp)) * (logkp_pred - np.mean(logkp_pred)))/(np.sqrt(np.sum((logkp - np.mean(logkp))**2))*np.sqrt(np.sum((logkp_pred - np.mean(logkp_pred))**2))))
# using Scipy
from scipy.stats import pearsonr
# also returns p-value that this correlation appeared by chance
print(pearsonr(logkp_pred, logkp))
# using Numpy
print(np.corrcoef(logkp_pred, logkp))